In [ ]:
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
# os.environ["TF_XLA_FLAGS=--tf_xla_enable_xla_devices"] = "false"

import tensorflow as tf

from train.models import FullModel
from util import dataset as u_dataset
from util import image as u_image

#### Load Test Data


In [ ]:
test_directory = "data/Joerg_Joerg_CompetitionWalk_GO2025__HULKs_2ndHalf_5"
label = u_dataset.load_labels(test_directory)[165]

camera = u_dataset.camera_from_label(label)
camera = tf.constant(camera, dtype=tf.float32)

intrinsics = u_dataset.intrinsics_from_label(label)
intrinsics = tf.constant(intrinsics, dtype=tf.float32)

image = u_dataset.load_image(test_directory, label, image_format=u_image.ImageFormat.YUYV)
image_yuv = tf.reshape(tf.constant(image), (-1, 480, 320, 4))

#### Load Encoder Model


In [ ]:
# Compile Full Model
model = FullModel(480, 320)
model.compile(optimizer=tf.keras.optimizers.Adam())

model.encoder.load_weights("./models/encoder/encoder_overfit.keras")

# Get Patch Extractor Layer from Full Model
patch_sampler = model.get_layer("patch_sampler")
patch_extractor = model.get_layer("patch_extractor")

#### Show Groundtruth Masks on Image


In [ ]:
u_image.show_cell_on_image(test_directory, label, "ball", grid_dims=(15, 20))

#### Show Predicted Patches


In [ ]:
results = model((image_yuv, camera, intrinsics), training=None)

u_image.show_patches_on_image(image, "ball", results)